# Paper figures

Generates the three data figures from files already in Drive. CPU is fine.

Outputs to `Drive/cotton_bench/figures/` as both PDF (for LaTeX) and PNG.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/cotton_bench'

import os, json
import numpy as np, pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

os.makedirs(f'{DRIVE}/figures', exist_ok=True)

# Consistent styling for a two-column paper
plt.rcParams.update({
    'font.size': 8, 'axes.labelsize': 8, 'axes.titlesize': 8,
    'xtick.labelsize': 7, 'ytick.labelsize': 7, 'legend.fontsize': 7,
    'figure.dpi': 200, 'savefig.bbox': 'tight', 'axes.grid': True,
    'grid.alpha': 0.3, 'grid.linewidth': 0.4,
})
LABEL = {'yolov5nu': 'YOLOv5n-u', 'yolo11n': 'YOLO11n', 'yolo11s': 'YOLO11s'}
COLOR = {'yolov5nu': '#4C72B0', 'yolo11n': '#DD8452', 'yolo11s': '#55A868'}

# ---------------------------------------------------------------- figure 1
fig, axes = plt.subplots(1, 2, figsize=(6.8, 2.4))
for run in ['yolov5nu', 'yolo11n', 'yolo11s']:
    p = f'{DRIVE}/baselines/{run}/results.csv'
    if not os.path.exists(p):
        print('missing', p); continue
    d = pd.read_csv(p)
    d.columns = [c.strip() for c in d.columns]
    axes[0].plot(d['epoch'], d['metrics/mAP50(B)'],
                 label=LABEL[run], color=COLOR[run], lw=1.2)
    axes[1].plot(d['epoch'], d['metrics/mAP50-95(B)'],
                 label=LABEL[run], color=COLOR[run], lw=1.2)
axes[0].set_xlabel('epoch'); axes[0].set_ylabel('mAP@50')
axes[1].set_xlabel('epoch'); axes[1].set_ylabel('mAP@[50:95]')
axes[0].legend(frameon=False); axes[1].legend(frameon=False)
fig.savefig(f'{DRIVE}/figures/fig_training_curves.pdf')
fig.savefig(f'{DRIVE}/figures/fig_training_curves.png')
print('wrote fig_training_curves')

# ---------------------------------------------------------------- figure 2
p = f'{DRIVE}/results_sweep.csv'
if os.path.exists(p):
    sweep = pd.read_csv(p)
    fig, ax = plt.subplots(figsize=(3.3, 2.4))
    for run, g in sweep.groupby('model'):
        g = g.sort_values('conf')
        ax.plot(g['conf'], g['F1'], label=LABEL.get(run, run),
                color=COLOR.get(run), lw=1.2)
        best = g.loc[g.F1.idxmax()]
        ax.plot(best['conf'], best['F1'], 'o', ms=3.5,
                color=COLOR.get(run), zorder=5)
    ax.set_xlabel('confidence threshold'); ax.set_ylabel('$F_1$')
    ax.legend(frameon=False, loc='lower center')
    fig.savefig(f'{DRIVE}/figures/fig_f1_sweep.pdf')
    fig.savefig(f'{DRIVE}/figures/fig_f1_sweep.png')
    print('wrote fig_f1_sweep')
else:
    print('missing results_sweep.csv, skipped F1 sweep')

# ---------------------------------------------------------------- figure 3
fig, ax = plt.subplots(figsize=(3.3, 2.4))
for c, colour, lab in [('pre', '#55A868', 'Pre-defoliation'),
                       ('post', '#C44E52', 'Post-defoliation')]:
    p = f'{DRIVE}/silver_{c}/raw_preds.json'
    if not os.path.exists(p):
        print('missing', p); continue
    counts = np.array([len(v) for v in json.load(open(p)).values()])
    ax.hist(counts, bins=np.arange(0, 700, 15), alpha=0.65,
            color=colour, label=f'{lab} (n={len(counts):,})')
    ax.axvline(np.median(counts), color=colour, ls='--', lw=1)
ax.set_xlabel('detections per tile'); ax.set_ylabel('tiles')
ax.set_xlim(0, 650)
ax.legend(frameon=False)
fig.savefig(f'{DRIVE}/figures/fig_density.pdf')
fig.savefig(f'{DRIVE}/figures/fig_density.png')
print('wrote fig_density')
print('\nall figures in', f'{DRIVE}/figures')